---

# **Transcrição por IA**
### Google Colab por André Trevisol Trindade

---

***Atualizado em 02/09/2026***

***O que é o Google Colab?***

O Google Colab é uma ferramenta gratuita do Google que permite escrever e executar código Python diretamente no navegador. É muito útil para análise de dados e aprendizado de máquina, oferecendo acesso a recursos computacionais poderosos, como GPUs e TPUs, sem nenhum custo. Além disso, permite colaboração em tempo real e fácil integração com o Google Drive.

***O que é o Whisper?***

O Whisper é uma ferramenta de inteligência artificial desenvolvida para transcrição de áudio. Ela converte automaticamente fala em texto com alta precisão. Ideal para transcrever entrevistas, reuniões, palestras e outros tipos de gravações de áudio, o Whisper facilita a criação de textos a partir de arquivos de áudio, economizando tempo e esforço.

📖 [Demonstração dos resultados](https://youtu.be/n-VhSvGEpTk)

> Qualquer dúvida, sugestão ou elogio me envie por e-mail em [**attrindade.dados@gmail.com**](mailto:attrindade.dados@gmail.com)
>
>**Considere fazer uma doação e ajudar a manter essa iniciativa, a chave Pix é o mesmo e-mail!**

---
<sub>Desenvolvido por [**André Trevisol Trindade**](https://www.attrindade.com) · Cientista de Dados com foco em pesquisa</sub>

<sub>Tecnologia: [WhisperX](https://github.com/m-bain/whisperX) · [faster-whisper](https://github.com/SYSTRAN/faster-whisper) · [pyannote.audio](https://github.com/pyannote/pyannote-audio) · [yt-dlp](https://github.com/yt-dlp/yt-dlp)</sub>

---

# PASSO 1 - Configurações

In [ ]:
# @markdown Clique em ▶ **play**.

import subprocess
import sys
from IPython.display import display, HTML, clear_output

_INSTALL_LOG = "install_passo1.txt"


def _barra(passo, total, descricao):
    nota = '<p style="margin:0 0 10px;font-size:12px;color:#d97706;font-style:italic">⚠️ Dica: Essa etapa deve levar cerca de 1 minuto. Não recarregue a página!</p>'
    display(
        HTML(
            "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
            'margin:8px 0;background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;padding:14px 18px">'
            '<p style="margin:0 0 8px;font-size:14px;font-weight:600;color:#1e293b">⏳ Preparando e Instalando… '
            f"({passo}/{total})</p>"
            f'<p style="margin:0 0 8px;font-size:13px;color:#475569">{descricao}</p>'
            f"{nota}"
            '<div style="background:#e2e8f0;border-radius:99px;height:6px">'
            f'<div style="background:#4f46e5;width:{int(passo/total*100)}%;height:6px;'
            'border-radius:99px"></div></div></div>'
        )
    )


def _pip(descricao, *pacotes):
    """Instala pacotes sem mostrar saída no terminal; grava tudo no log."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--prefer-binary", *pacotes],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    with open(_INSTALL_LOG, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*50}\n{descricao}\n{'='*50}\n")
        if result.stdout.strip():
            f.write(result.stdout)
        if result.stderr.strip():
            f.write("[stderr]\n" + result.stderr)
    return result.returncode


if "passo1_concluido" not in globals():
    # Limpa log anterior de instalação (se houver)
    open(_INSTALL_LOG, "w").close()

    _barra(1, 4, "Dependências do WhisperX (faster-whisper, transformers, pyannote…)")
    _pip(
        "(1/3) Dependências do WhisperX",
        "faster-whisper",
        "ctranslate2",
        "transformers",
        "pyannote.audio==3.1.1",
        "pyannote.core<6.0",
        "pyannote.database<6.0",
        "pyannote.metrics<4.0",
        "pyannote.pipeline<4.0",
    )

    clear_output(wait=True)
    _barra(2, 4, "WhisperX (sem forçar versão do torch)")
    _pip("(2/3) WhisperX", "whisperx", "--no-deps")

    clear_output(wait=True)
    _barra(3, 4, "Utilitários do projeto (pydub, gdown, yt-dlp…)")
    _pip(
        "(3/3) Utilitários",
        "pydub",
        "gdown==5.2.0",
        "loguru",
        "nltk",
        "omegaconf",
        "yt-dlp",
        "lameenc>=1.2",
        "openunmix",
        "wget",
    )

    globals()["passo1_concluido"] = True
    clear_output(wait=True)

    import site
    import importlib

    importlib.reload(site)


# ---------------------------------------------------------------------------
# IMPORTAÇÕES RÁPIDAS — necessárias antes do card de progresso
# ---------------------------------------------------------------------------
from IPython.display import clear_output, display, Markdown, HTML
from google.colab import output, files

display(
    HTML(
        "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
        'margin:8px 0;background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;padding:14px 18px">'
        '<p style="margin:0 0 8px;font-size:14px;font-weight:600;color:#1e293b">⏳ Preparando e Instalando… (4/4)</p>'
        '<p style="margin:0 0 8px;font-size:13px;color:#475569">Carregando funções internas…</p>'
        '<p style="margin:0 0 10px;font-size:12px;color:#d97706;font-style:italic">⚠️ Dica: Essa etapa deve levar cerca de 1 minuto. Não recarregue a página!</p>'
        '<div style="background:#e2e8f0;border-radius:99px;height:6px">'
        '<div style="background:#4f46e5;width:80%;height:6px;border-radius:99px"></div></div></div>'
    )
)

# ---------------------------------------------------------------------------
# IMPORTAÇÕES PESADAS
# ---------------------------------------------------------------------------
import gc
import gdown
import json
import os
import re
import requests
import shutil
import subprocess
import sys
import datetime
import warnings
import pytz
import torch

# Suprime as barras de progresso de download do HuggingFace Hub (tqdm)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Fix pyannote.audio 3.1.1 compatibility with colab's newer torchaudio
import sys
import types
import torchaudio

if not hasattr(torchaudio, "AudioMetaData"):
    torchaudio.AudioMetaData = type("AudioMetaData", (), {})
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = lambda backend: None
if not hasattr(torchaudio, "get_audio_backend"):
    torchaudio.get_audio_backend = lambda: "soundfile"

if "torchaudio.backend" not in sys.modules:
    _backend_mod = types.ModuleType("torchaudio.backend")
    _common_mod = types.ModuleType("torchaudio.backend.common")
    _common_mod.AudioMetaData = torchaudio.AudioMetaData
    _backend_mod.common = _common_mod
    sys.modules["torchaudio.backend"] = _backend_mod
    sys.modules["torchaudio.backend.common"] = _common_mod
    torchaudio.backend = _backend_mod

# Fix huggingface_hub crashing when pyannote.audio passes 'use_auth_token'
import huggingface_hub

_orig_hf_hub_download = huggingface_hub.hf_hub_download


def _patched_hf_hub_download(*args, **kwargs):
    if "use_auth_token" in kwargs:
        kwargs["token"] = kwargs.pop("use_auth_token")
    return _orig_hf_hub_download(*args, **kwargs)


huggingface_hub.hf_hub_download = _patched_hf_hub_download

_orig_ModelCard_load = huggingface_hub.ModelCard.load


@classmethod
def _patched_ModelCard_load(cls, *args, **kwargs):
    if "use_auth_token" in kwargs:
        kwargs["token"] = kwargs.pop("use_auth_token")
    return _orig_ModelCard_load.__func__(cls, *args, **kwargs)


huggingface_hub.ModelCard.load = _patched_ModelCard_load

# Fix numpy internal C-extension symbols and NumPy 2.0 expired attributes
try:
    import numpy as _np
    _compat_map = {
        "NaN": _np.nan,
        "NAN": _np.nan,
        "Inf": _np.inf,
        "Infinity": _np.inf,
        "infty": _np.inf,
        "PINF": _np.inf,
        "NINF": -_np.inf,
        "float_": _np.float64,
        "complex_": _np.complex128,
        "string_": _np.bytes_,
        "unicode_": _np.str_,
        "asfarray": _np.asarray,
        "alltrue": _np.all,
        "sometrue": _np.any,
        "round_": _np.round,
    }
    for _k, _v in _compat_map.items():
        setattr(_np, _k, _v)

    _orig_np_getattr = getattr(_np, "__getattr__", None)
    if _orig_np_getattr is not None:
        def _safe_np_getattr(attr):
            if attr in _compat_map:
                return _compat_map[attr]
            try:
                return _orig_np_getattr(attr)
            except AttributeError:
                if hasattr(_np, "__expired_attributes__") and attr in _np.__expired_attributes__:
                    _target = _np.__expired_attributes__[attr]
                    _m = re.search(r"Use `np\.([^`]+)`", _target)
                    if _m and hasattr(_np, _m.group(1)):
                        _val = getattr(_np, _m.group(1))
                        setattr(_np, attr, _val)
                        return _val
                raise
        _np.__getattr__ = _safe_np_getattr
except Exception:
    pass

try:
    import numpy._core.umath as _numath
    for _attr in ("_slice", "_center", "_expandtabs"):
        if not hasattr(_numath, _attr):
            setattr(_numath, _attr, lambda *args, **kwargs: None)
except Exception:
    pass

try:
    import numpy._core._multiarray_umath as _nmumath
    if not hasattr(_nmumath, "_blas_supports_fpe"):
        _nmumath._blas_supports_fpe = lambda *args, **kwargs: True
except Exception:
    pass

# Fix pyannote.audio expecting 'use_auth_token' while WhisperX passes 'token'
import pyannote.audio

_original_from_pretrained = pyannote.audio.Pipeline.from_pretrained


@classmethod
def _patched_from_pretrained(cls, checkpoint, **kwargs):
    if "token" in kwargs:
        kwargs["use_auth_token"] = kwargs.pop("token")
    return _original_from_pretrained.__func__(cls, checkpoint, **kwargs)


pyannote.audio.Pipeline.from_pretrained = _patched_from_pretrained

warnings.filterwarnings("ignore", category=SyntaxWarning, module="pydub")
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

# Suprime avisos de autenticação do huggingface_hub que aparecem via logging
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

import whisperx
from pydub import AudioSegment  # noqa: F401 — usado por whisperx internamente
from urllib.parse import unquote
from loguru import logger

# ---------------------------------------------------------------------------
# LOG DE SESSÃO
# Um arquivo por sessão, com timestamp no nome para não sobrescrever sessões
# anteriores. Nível DEBUG garante que tudo é capturado.
# ---------------------------------------------------------------------------

_SESSION_START = datetime.datetime.now(pytz.timezone("America/Sao_Paulo"))
_SESSION_ID = _SESSION_START.strftime("%Y%m%d_%H%M%S")
LOG_FILE = f"sessao_{_SESSION_ID}.log"

logger.remove()
logger.add(
    LOG_FILE,
    format="{time:YYYY-MM-DD HH:mm:ss.SSS} | {level:<8} | {function}:{line} | {message}",
    level="DEBUG",
    encoding="utf-8",
    rotation=None,  # nunca rotaciona — queremos tudo numa sessão só
    retention=5,  # mantém no máximo 5 arquivos de sessões anteriores
)

# ---------------------------------------------------------------------------
# INTEGRAÇÃO DO LOG DE INSTALAÇÃO (Passo 1)
# O Passo 1 grava a saída do pip num arquivo temporário.
# Aqui incorporamos esse conteúdo ao log principal e apagamos o temporário.
# ---------------------------------------------------------------------------

_INSTALL_LOG_PATH = "install_passo1.txt"
if os.path.exists(_INSTALL_LOG_PATH):
    try:
        _install_txt = open(_INSTALL_LOG_PATH, encoding="utf-8").read().strip()
        if _install_txt:
            logger.info("=== Saída do Passo 1 (instalação) ===")
            logger.info(_install_txt)
            logger.info("=== Fim do log de instalação ===")
        os.remove(_INSTALL_LOG_PATH)
    except Exception as _e:
        logger.warning(f"Não foi possível ler o log de instalação: {_e}")

# ---------------------------------------------------------------------------
# CABEÇALHO DE SESSÃO — informações do ambiente para diagnóstico
# ---------------------------------------------------------------------------


def _log_cabecalho_sessao():
    sep = "=" * 65
    logger.info(sep)
    logger.info("INÍCIO DE SESSÃO — Transcrição por IA (WhisperX)")
    logger.info(sep)
    logger.info(f"Sessão ID  : {_SESSION_ID}")
    logger.info(
        f"Horário    : {_SESSION_START.strftime('%H:%M:%S de %d/%m/%Y')} (Brasília)"
    )
    logger.info(f"Python     : {sys.version.split()[0]}")
    logger.info(f"PyTorch    : {torch.__version__}")
    cuda_ok = torch.cuda.is_available()
    logger.info(
        f"CUDA       : {'disponível' if cuda_ok else 'INDISPONÍVEL — rodando em CPU'}"
    )
    if cuda_ok:
        logger.info(f"GPU        : {torch.cuda.get_device_name(0)}")
        logger.info(f"CUDA versão: {torch.version.cuda}")
        mem_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"VRAM total : {mem_total:.1f} GB")
    try:
        logger.info(f"WhisperX   : {whisperx.__version__}")
    except AttributeError:
        logger.info("WhisperX   : instalado (versão não exposta)")
    try:
        runtime = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            timeout=5,
        )
        if runtime.returncode == 0:
            logger.info(f"Driver GPU : {runtime.stdout.strip()}")
    except Exception:
        pass
    logger.info(sep)


_log_cabecalho_sessao()

# ---------------------------------------------------------------------------
# CAPTURA GLOBAL DE EXCEÇÕES
# ---------------------------------------------------------------------------


def _registrar_excecao(shell, etype, evalue, tb, tb_offset=None):
    import traceback

    tb_str = "".join(traceback.format_exception(etype, evalue, tb))
    logger.error(f"EXCEÇÃO NÃO TRATADA: {etype.__name__}: {evalue}")
    logger.error(f"Traceback completo:\n{tb_str}")
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


ip = get_ipython()
if ip:
    ip.set_custom_exc((Exception,), _registrar_excecao)

# ---------------------------------------------------------------------------
# ESTADO GLOBAL DE ETAPAS
# ---------------------------------------------------------------------------

if "etapas_concluidas" not in globals():
    globals()["etapas_concluidas"] = {1}

logger.info("Ambiente inicializado. Pronto para o Passo 2.")


PASSOS_2 = {'2a', '2b', '2c'}

# ---------------------------------------------------------------------------
# DESIGN SYSTEM — cards HTML reutilizáveis
# Cada função exibe um card visual no output do Colab.
# ---------------------------------------------------------------------------

_CARD_CSS = (
    "font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
    "margin:10px 0;border-radius:10px;padding:16px 20px;"
    "box-shadow:0 1px 4px rgba(0,0,0,.08);line-height:1.5"
)
_TITLE_CSS = "margin:0 0 5px;font-size:17px;font-weight:700;color:#1e293b"
_BODY_CSS  = "margin:0;font-size:14px;color:#475569;line-height:1.55"
_META_CSS  = "margin:8px 0 0;font-size:12px;color:#94a3b8"

_TEMAS = {
    'ok':   ('#d1fae5', '#059669', '✅'),
    'info': ('#dbeafe', '#2563eb', 'ℹ️'),
    'warn': ('#fef9c3', '#d97706', '⚠️'),
    'err':  ('#fee2e2', '#dc2626', '❌'),
    'load': ('#f0f9ff', '#0284c7', '⏳'),
}


def _card_html(tipo, titulo, corpo, meta='', extra=''):
    bg, borda, icone = _TEMAS[tipo]
    meta_html = f'<p style="{_META_CSS}">{meta}</p>' if meta else ''
    return (
        f'<div style="{_CARD_CSS};background:{bg};border-left:4px solid {borda}">'
        f'<p style="{_TITLE_CSS}">{icone}&nbsp; {titulo}</p>'
        f'<p style="{_BODY_CSS}">{corpo}</p>'
        f'{meta_html}{extra}'
        f'</div>'
    )


def card_ok(titulo, corpo='', meta=''):
    display(HTML(_card_html('ok', titulo, corpo, meta)))

def card_info(titulo, corpo='', meta=''):
    display(HTML(_card_html('info', titulo, corpo, meta)))

def card_aviso(titulo, corpo='', meta=''):
    display(HTML(_card_html('warn', titulo, corpo, meta)))

def card_erro(titulo, corpo='', meta=''):
    display(HTML(_card_html('err', titulo, corpo, meta)))

def card_aguarde(titulo, corpo=''):
    display(HTML(_card_html('load', titulo, corpo)))


# ---------------------------------------------------------------------------
# SILENCIADOR DE OUTPUTS
# ---------------------------------------------------------------------------
import io
import sys

import re

class Silenciador:
    def __init__(self, display_id=None):
        self.display_id = display_id
        self.last_pct = -1

    def __enter__(self):
        self._stdout = io.StringIO()
        self._stderr = io.StringIO()
        self.old_stdout = sys.stdout
        self.old_stderr = sys.stderr

        class OutInterceptor:
            def __init__(self, silencer, is_stderr=False):
                self.s = silencer
                self.is_stderr = is_stderr
                self.buf = ""
                # Referência ao stream original para isatty/fileno/flush
                self.original = silencer.old_stderr if is_stderr else silencer.old_stdout

            def write(self, text):
                if self.is_stderr:
                    self.s._stderr.write(text)
                else:
                    self.s._stdout.write(text)

            def flush(self):
                if hasattr(self.original, 'flush'):
                    self.original.flush()

            def isatty(self):
                # tqdm e transformers precisam disso para renderizar
                if hasattr(self.original, 'isatty'):
                    return self.original.isatty()
                return False

            def fileno(self):
                if hasattr(self.original, 'fileno'):
                    return self.original.fileno()
                raise io.UnsupportedOperation("fileno")

        sys.stdout = OutInterceptor(self, is_stderr=False)
        sys.stderr = OutInterceptor(self, is_stderr=True)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout = self.old_stdout
        sys.stderr = self.old_stderr
        
        try:
            from loguru import logger
            out_text = self._stdout.getvalue().strip()
            err_text = self._stderr.getvalue().strip()
            if out_text:
                logger.debug(f"[STDOUT]\n{out_text}")
            if err_text:
                logger.debug(f"[STDERR]\n{err_text}")
        except:
            pass

# ---------------------------------------------------------------------------
# TIMESTAMP E CONTROLE DE ETAPAS
# ---------------------------------------------------------------------------

def obter_timestamp_brasil() -> str:
    """Retorna o horário atual no fuso de Brasília."""
    tz = pytz.timezone('America/Sao_Paulo')
    return datetime.datetime.now(tz).strftime("%H:%M:%S de %d/%m/%Y")


def verificar_etapas(passo_atual):
    """
    Verifica se os pré-requisitos do passo foram cumpridos.
    Levanta RuntimeError em vez de sys.exit (que poderia matar o kernel).
    """
    prerequisitos = {
        1:     [],
        '2a':  [1],
        '2b':  [1],
        '2c':  [1],
        '3.1': [1, '2'],
        '3.2': [1, '2', '3.1'],
        4:     [1, '2', '3.1'],
    }

    if passo_atual == 1:
        return

    concluidas = globals()['etapas_concluidas']

    if passo_atual not in prerequisitos:
        raise ValueError(f"Passo '{passo_atual}' não reconhecido.")

    # Novo áudio: reseta estado se já havia uma transcrição em andamento
    if passo_atual in PASSOS_2 and concluidas & (PASSOS_2 | {'3.1', '3.2', 4}):
        card_aviso(
            'Nova transcrição detectada',
            'O estado anterior foi resetado. Prossiga normalmente.',
        )
        _reset_para_nova_transcricao()
        concluidas = globals()['etapas_concluidas']

    for req in prerequisitos[passo_atual]:
        if req == '2':
            if not concluidas & PASSOS_2:
                raise RuntimeError("Execute um dos Passos 2 (upload ou link) antes de prosseguir.")
        elif req not in concluidas:
            raise RuntimeError(f"Execute o Passo {req} antes de prosseguir.")


def _reset_para_nova_transcricao():
    """Remove variáveis da transcrição anterior para iniciar uma nova."""
    concluidas = globals().get('etapas_concluidas', {1})
    globals()['etapas_concluidas'] = concluidas - (PASSOS_2 | {'3.1', '3.2', 4})
    for var in ('result', 'ssm', 'ssm_mod', 'audio', 'vocal_target', 'vocal_target_filename'):
        globals().pop(var, None)
    globals()['batch_jobs'] = []


_CHUNK_SIZE = 8 * 1024 * 1024  # 8 MB — equilibra memória e velocidade de download
_REQUEST_TIMEOUT = 60           # segundos

# ---------------------------------------------------------------------------
# Coleta de texto via formulário inline (substitui input() )
# ---------------------------------------------------------------------------

def _coletar_input_js(titulo, instrucao, placeholder, multilinha=False):
    """
    Exibe um formulário inline no output do Colab e retorna o valor digitado.
    Totalmente seguro contra caracteres especiais.
    """
    params = {
        "titulo": titulo,
        "instrucao": instrucao,
        "placeholder": placeholder,
        "multilinha": multilinha,
        "inp_css": (
            "width:100%;box-sizing:border-box;padding:10px 14px;"
            "border:1.5px solid #cbd5e1;border-radius:8px;font-size:14px;"
            "color:#1e293b;background:#fff;outline:none;"
            "font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
        ),
        "btn_css": (
            "background:#4f46e5;color:#fff;border:none;border-radius:8px;"
            "padding:10px 24px;font-size:14px;font-weight:600;cursor:pointer;"
            "font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
            "margin-top:10px"
        )
    }

    js = f"""
(async () => {{
  const params = {json.dumps(params)};
  
  let container = document.getElementById('attrindade-ui-container');
  if(!container) {{
    container = document.createElement('div');
    container.id = 'attrindade-ui-container';
    document.body.appendChild(container);
  }}
  container.innerHTML = ''; // Limpa qualquer resquício

  const wrapper = document.createElement('div');
  wrapper.style.cssText = `
    font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
    margin:10px 0;background:#eff6ff;border-left:4px solid #2563eb;
    border-radius:10px;padding:18px 22px
  `;

  const inpHtml = params.multilinha
    ? `<textarea id="wb-inp" rows="3" placeholder="${{params.placeholder.replace(/"/g, '&quot;')}}" style="${{params.inp_css}} resize:vertical;min-height:70px;margin-bottom:10px;display:block"></textarea>`
    : `<input id="wb-inp" type="text" placeholder="${{params.placeholder.replace(/"/g, '&quot;')}}" style="${{params.inp_css}} margin-bottom:0;display:block" />`;

  wrapper.innerHTML = `
    <p style="margin:0 0 4px;font-size:17px;font-weight:700;color:#1e293b">ℹ️ ${{params.titulo}}</p>
    <p style="margin:0 0 12px;font-size:14px;color:#475569">${{params.instrucao}}</p>
    ${{inpHtml}}
    <div style="display:flex;gap:10px">
      <button id="wb-btn" style="${{params.btn_css}}">Continuar →</button>
      <button id="wb-voltar" style="background:#e2e8f0;color:#475569;border:none;border-radius:8px;padding:10px 24px;font-size:14px;font-weight:600;cursor:pointer;transition:background 0.2s">← Voltar</button>
    </div>
    <p id="wb-err" style="margin:8px 0 0;font-size:13px;color:#dc2626;display:none">
      Por favor, preencha o campo acima antes de continuar.
    </p>
  `;
  container.appendChild(wrapper);

  const inp = wrapper.querySelector('#wb-inp');
  inp.focus();

  return new Promise((resolve) => {{
    function go() {{
      const val = inp.value.trim();
      if(!val) {{
        wrapper.querySelector('#wb-err').style.display='block';
        return;
      }}
      wrapper.innerHTML = '<p style="font-size:14px;color:#059669;font-weight:600;margin:0">✅ Recebido!</p>';
      setTimeout(() => container.innerHTML = '', 1000);
      resolve(val);
    }}
    wrapper.querySelector('#wb-btn').onclick = go;
    wrapper.querySelector('#wb-voltar').onclick = () => {{
      container.innerHTML = '';
      resolve('__VOLTAR__');
    }};
    inp.onkeydown = (e) => {{
      if(e.key==='Enter' && !e.shiftKey) go();
    }};
  }});
}})()
"""
    return output.eval_js(js)


# ---------------------------------------------------------------------------
# Helpers de download
# ---------------------------------------------------------------------------

def move_para_pasta_input(nome_arquivo: str) -> str:
    """Move o arquivo baixado para a pasta input/ e retorna o novo caminho."""
    os.makedirs('input', exist_ok=True)
    destino = os.path.join('input', os.path.basename(nome_arquivo))
    shutil.move(nome_arquivo, destino)
    return destino


def _extrair_nome_do_header(content_disposition: str) -> str | None:
    """Tenta extrair o nome do arquivo a partir do header Content-Disposition."""
    match = re.search(r"filename\*=.*''(.+)", content_disposition)
    if match:
        return unquote(match.group(1))
    match = re.search(r'filename="?([^";]+)"?', content_disposition)
    if match:
        return match.group(1)
    return None


def baixar_arquivo(link: str) -> str:
    """
    Faz download de uma URL e salva o arquivo localmente.
    Levanta RuntimeError em caso de falha HTTP.
    """
    resposta = requests.get(link, stream=True, timeout=_REQUEST_TIMEOUT)

    if resposta.status_code != 200:
        raise RuntimeError(f"Falha no download (HTTP {resposta.status_code}): {link}")

    cd = resposta.headers.get('Content-Disposition', '')
    nome_arquivo = _extrair_nome_do_header(cd) or "arquivo_baixado"

    with open(nome_arquivo, 'wb') as f:
        for chunk in resposta.iter_content(chunk_size=_CHUNK_SIZE):
            f.write(chunk)

    return nome_arquivo


def _baixar_onedrive(embed_url: str):
    """Processa o embed do OneDrive e retorna (caminho, nome_arquivo)."""
    match = re.search(r'src="([^"]+)"', embed_url)
    if not match:
        raise ValueError(
            "Texto do OneDrive inválido. Use o texto gerado pelo botão 'Embed' do arquivo."
        )
    download_url = match.group(1).replace("embed", "download")
    nome = baixar_arquivo(download_url)
    return move_para_pasta_input(nome), nome


def insira_link(site: str):
    """
    Coleta o link do usuário via formulário inline e faz o download.
    Retorna (vocal_target, vocal_target_filename).
    """
    if site == "Google Drive":
        logger.info("Google Drive selecionado.")
        link = _coletar_input_js(
            titulo="Cole o link do Google Drive",
            instrucao=(
                "Abra o Google Drive → clique com botão direito no arquivo → "
                "<strong>Compartilhar</strong> → <strong>Qualquer pessoa com o link</strong> → "
                "<strong>Copiar link</strong>."
            ),
            placeholder="https://drive.google.com/file/d/...",
        )
        if not link:
            raise RuntimeError("Nenhum link foi informado.")
        if link == '__VOLTAR__':
            return '__VOLTAR__', '__VOLTAR__'
        
        logger.info(f"Link recebido: {link}")
        card_aguarde("Baixando do Google Drive...", "Isso pode levar alguns segundos dependendo do tamanho do arquivo.")
        try:
            # Em alguns casos, gdown precisa de quiet=False para mostrar erros no terminal (log)
            # fuzzy=True ajuda a encontrar o ID em links de compartilhamento
            nome = gdown.download(url=link, fuzzy=True, quiet=False, use_cookies=False)
            
            if not nome:
                # Tentativa secundária: talvez seja um link direto?
                logger.warning("gdown retornou None. Tentando novamente sem fuzzy...")
                nome = gdown.download(url=link, quiet=False, use_cookies=False)
                
            if not nome:
                 raise RuntimeError("Não foi possível baixar o arquivo. Verifique se o link está público.")
            
        except Exception as e:
            logger.error(f"Erro no gdown: {e}")
            card_erro(
                'Não consegui baixar o arquivo do Google Drive',
                'Verifique se o arquivo está compartilhado como '
                '<strong>"Qualquer pessoa com o link"</strong> e se o link está correto.',
            )
            raise
        return move_para_pasta_input(nome), os.path.basename(nome)

    elif site == "OneDrive":
        logger.info("OneDrive selecionado.")
        embed = _coletar_input_js(
            titulo="Cole o código de incorporação do OneDrive",
            instrucao=(
                "No OneDrive, clique com botão direito no arquivo → "
                "<strong>Incorporar</strong> (Embed) → copie <em>todo</em> o texto gerado."
            ),
            placeholder='<iframe src="https://onedrive.live.com/embed?..." ...></iframe>',
            multilinha=True,
        )
        if not embed:
            raise RuntimeError("Nenhum texto foi informado.")
        if embed == '__VOLTAR__':
            return '__VOLTAR__', '__VOLTAR__'
        card_aguarde("Baixando do OneDrive...", "Aguarde enquanto processamos o link.")
        return _baixar_onedrive(embed)

    elif site == "DropBox":
        logger.info("Dropbox selecionado.")
        link = _coletar_input_js(
            titulo="Cole o link do Dropbox",
            instrucao=(
                "No Dropbox, clique com botão direito no arquivo → "
                "<strong>Copiar link</strong>."
            ),
            placeholder="https://www.dropbox.com/s/...",
        )
        if not link:
            raise RuntimeError("Nenhum link foi informado.")
        if link == "__VOLTAR__":
            return "__VOLTAR__", "__VOLTAR__"
        url_download = link.replace("?dl=0", "?dl=1").replace("?rlkey=", "?dl=1&rlkey=")
        card_aguarde("Baixando do DropBox...", "O download começará em instantes.")
        try:
            nome = baixar_arquivo(url_download)
        except Exception:
            card_erro(
                "Não consegui baixar o arquivo do Dropbox",
                "Verifique se o link está correto e se o arquivo é público.",
            )
            raise
        return move_para_pasta_input(nome), os.path.basename(nome)

    elif site == "YouTube / Link":
        logger.info("YouTube / Link Externo selecionado.")
        link = _coletar_input_js(
            titulo="Cole o link do vídeo ou áudio",
            instrucao=(
                "Suporta YouTube, Instagram, Facebook, Twitter e centenas de outros sites."
            ),
            placeholder="https://www.youtube.com/watch?v=...",
        )
        if not link:
            raise RuntimeError("Nenhum link foi informado.")
        if link == "__VOLTAR__":
            return "__VOLTAR__", "__VOLTAR__"

        card_aguarde("Obtendo informações do vídeo...", "Estamos preparando o download do áudio.")
        try:
            # 1. Pegar título
            res_title = subprocess.run(
                ["yt-dlp", "--get-title", "--no-warnings", link],
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
            )
            nome_base = res_title.stdout.strip() or "video_youtube"
            nome_sanitizado = re.sub(r'[\\/:*?"<>|]', "", nome_base)[:50]

            # 2. Baixar áudio
            card_aguarde(f"Baixando: {nome_base}...", "Isso pode levar alguns minutos se o vídeo for longo. Não feche a aba.")
            logger.info(f"Baixando: {nome_base}")
            subprocess.run(
                [
                    "yt-dlp",
                    "-x",
                    "--audio-format",
                    "mp3",
                    "--audio-quality",
                    "0",
                    "-o",
                    f"{nome_sanitizado}.%(ext)s",
                    link,
                ],
                check=True,
            )
            vocal_filename = f"{nome_sanitizado}.mp3"
            return move_para_pasta_input(vocal_filename), vocal_filename
        except Exception as e:
            logger.error(f"Erro no yt-dlp: {e}")
            card_erro(
                "Não consegui baixar o link informado",
                "Verifique se o link está acessível publicamente ou se o site é suportado.",
            )
            raise

    else:
        raise ValueError(f"Serviço '{site}' não suportado.")


import json
try:
    from google.colab import output
except ImportError:
    output = None
from loguru import logger



def name_speakers_async(batch_dict: dict, batch_jobs_ref: list):
    """
    Exibe um formulário inline para nomear cada locutor identificado em vários arquivos.
    Não bloqueia a execução da célula! Registra um callback para ser chamado pelo JS
    quando o usuário confirmar os nomes.
    """
    from IPython.display import display, HTML
    
    all_cards_html = ""
    has_any = False
    
    for filename, segments in batch_dict.items():
        # Coleta até 3 trechos por locutor
        speaker_texts = {}
        for seg in segments:
            speaker = seg.get("speaker")
            if not speaker:
                continue
            text = seg.get("text", "").strip()
            if not text:
                continue
            ts = format_timestamp(seg["start"], exclude_miliseconds=True)
            if speaker not in speaker_texts:
                speaker_texts[speaker] = []
            if len(speaker_texts[speaker]) < 3:
                speaker_texts[speaker].append(f'<span style="color:#94a3b8;font-size:12px">{ts}</span>&nbsp; "{text[:90]}"')

        if not speaker_texts:
            continue
            
        has_any = True
        
        cards_html = f'<div style="margin-bottom:20px;padding-top:15px;border-top:2px dashed #cbd5e1"><p style="font-weight:700;font-size:15px;color:#334155;margin-bottom:12px">📄 {filename}</p>'
        ids = sorted(speaker_texts.keys())
        total = len(ids)
        
        for i, id_loc in enumerate(ids):
            trechos_html = "".join([
                f'<div style="font-size:13px;color:#475569;padding:5px 0;border-bottom:1px solid #f1f5f9;line-height:1.45">{t}</div>'
                for t in speaker_texts[id_loc]
            ])
            
            import html
            fname_enc = html.escape(filename)
            
            cards_html += f"""
            <div style="background:#fff;border:1px solid #e2e8f0;border-radius:10px;padding:16px 20px;margin-bottom:12px;box-shadow:0 1px 3px rgba(0,0,0,.06);margin-left:10px">
                <p style="margin:0 0 10px;font-size:11px;font-weight:700;color:#94a3b8;text-transform:uppercase;letter-spacing:.05em">Pessoa {i+1} de {total}</p>
                <div style="margin-bottom:12px">{trechos_html}</div>
                <input data-file="{fname_enc}" data-id="{id_loc}" type="text" placeholder="Nome desta pessoa (ex: João, Entrevistadora…)"
                       style="width:100%;box-sizing:border-box;padding:9px 13px;border:1.5px solid #cbd5e1;border-radius:7px;font-size:14px;color:#1e293b;background:#f8fafc;outline:none;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif" />
            </div>
            """
        cards_html += "</div>"
        all_cards_html += cards_html

    if not has_any:
        return

    # O Callback Python que o JS vai chamar
    def on_speakers_confirmed(result_json_str):
        import json
        speaker_names_dict = json.loads(result_json_str)
        
        # Log resumo
        for fname, names in speaker_names_dict.items():
            logger.info(f"[Diarização] {fname}")
            for orig, nome in names.items():
                logger.info(f"  {orig} → {nome}")
                
        # Atualiza os batch_jobs
        for job in batch_jobs_ref:
            if not job.get("diarization_temp"):
                continue
            fname = job["filename"]
            names = speaker_names_dict.get(fname, {})
            # Aplica a atualização definindo no_result o que está salvo em temp
            for seg in job["diarization_temp"]:
                original = seg.get("speaker", "")
                if original in names:
                    seg["speaker"] = names[original]
            job["diarization_result"] = job["diarization_temp"]
            
        logger.info("Nomes de locutores atualizados em todos os arquivos no estado background.")

    if output is not None:
        output.register_callback('wb_save_speakers', on_speakers_confirmed)

    full_html = f"""
    <div id="wb-speaker-wrapper" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin:10px 0;background:#f8fafc;padding:15px;border-radius:10px">
        <div style="background:#eff6ff;border-left:4px solid #2563eb;border-radius:10px;padding:16px 20px;margin-bottom:16px">
            <p style="margin:0 0 4px;font-size:17px;font-weight:700;color:#1e293b">👥&nbsp; Identificar Locutores</p>
            <p style="margin:0;font-size:14px;color:#475569">Veja os trechos separados por arquivo e escreva o nome de cada pessoa. <strong>O modelo já terminou de rodar, então você pode demorar o tempo que precisar.</strong></p>
        </div>
        {all_cards_html}
        <button id="wb-confirm" style="background:#4f46e5;color:#fff;border:none;border-radius:8px;padding:11px 28px;font-size:15px;font-weight:600;cursor:pointer;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;width:100%;margin-top:4px">Confirmar nomes →</button>
        <p id="wb-name-err" style="margin:10px 0 0;font-size:13px;color:#dc2626;display:none">Preencha o nome de todas as pessoas antes de continuar.</p>
    </div>
    <script>
    (function() {{
      const wrapper = document.getElementById('wb-speaker-wrapper');
      function confirm() {{
        const inputs = wrapper.querySelectorAll('input[data-id]');
        const result = {{}};
        let ok = true;
        inputs.forEach(inp => {{
          const v = inp.value.trim();
          if (!v) ok = false;
          
          const file = inp.getAttribute('data-file');
          const id = inp.getAttribute('data-id');
          if(!result[file]) result[file] = {{}};
          result[file][id] = v || id;
        }});
        if (!ok) {{
          document.getElementById('wb-name-err').style.display = 'block';
          return;
        }}
        
        wrapper.innerHTML = `
          <div style="background:#d1fae5;border-left:4px solid #059669;border-radius:10px;padding:14px 18px">
            <p style="margin:0 0 4px;font-size:15px;font-weight:700;color:#1e293b">✅&nbsp; Nomes confirmados e salvos!</p>
            <p style="margin:0;font-size:13px;color:#475569">Prossiga para o <strong>Passo 4</strong> abaixo.</p>
          </div>
        `;
        // Envia de volta para o Python sem bloquear
        google.colab.kernel.invokeFunction('wb_save_speakers', [JSON.stringify(result)], {{}});
      }}

      document.getElementById('wb-confirm').addEventListener('click', confirm);
      wrapper.querySelectorAll('input[data-id]').forEach(inp => {{
        inp.addEventListener('keydown', (e) => {{ if(e.key === 'Enter') confirm(); }});
      }});
    }})();
    </script>
    """
    
    display(HTML(full_html))

def update_speaker_names(segments: list, speaker_names: dict) -> list:
    """Substitui os IDs de locutor pelos nomes fornecidos pelo usuário."""
    for seg in segments:
        original = seg.get("speaker", "")
        if original in speaker_names:
            seg["speaker"] = speaker_names[original]
    return segments


def format_timestamp(
    seconds: float,
    always_include_hours: bool = False,
    decimal_marker: str = ".",
    exclude_miliseconds: bool = False,
) -> str:
    assert seconds >= 0, "Timestamp não pode ser negativo."
    hours = int(seconds // 3600)
    seconds -= hours * 3600
    minutes = int(seconds // 60)
    seconds -= minutes * 60
    millis = int((seconds % 1) * 1000)
    seconds = int(seconds)

    h = f"{hours:02d}:" if always_include_hours or hours > 0 else ""
    if exclude_miliseconds:
        return f"{h}{minutes:02d}:{seconds:02d}"
    return f"{h}{minutes:02d}:{seconds:02d}{decimal_marker}{millis:03d}"


def write_txt(
    segments: list, f, locutores: bool = False, timestamp_linha: bool = False
):
    """Escreve a transcrição em texto simples."""
    prev_speaker = None

    for seg in segments:
        texto = seg["text"].strip()
        if not texto:
            continue
        ts = format_timestamp(seg["start"], exclude_miliseconds=True)
        speaker = seg.get("speaker", "")

        if locutores and speaker != prev_speaker:
            f.write(f"\n{speaker} [{ts}]:\n")
            prev_speaker = speaker

        if timestamp_linha:
            f.write(f"[{ts}] {texto}\n")
        else:
            f.write(texto + "\n")


def write_srt(segments: list, f, locutores: bool = False):
    """Escreve a transcrição em formato SRT."""
    for i, seg in enumerate(segments, start=1):
        inicio = format_timestamp(
            seg["start"], always_include_hours=True, decimal_marker=","
        )
        fim = format_timestamp(
            seg["end"], always_include_hours=True, decimal_marker=","
        )
        texto = seg["text"].strip().replace("-->", "->")
        if not texto:
            continue
        linha = f"{seg.get('speaker', '')}: {texto}" if locutores else texto
        f.write(f"{i}\n{inicio} --> {fim}\n{linha}\n\n")


def write_nvivo_tabbed(segments: list, f):
    """Escreve a transcrição em formato tabulado compatível com NVivo."""

    def _fmt(s):
        m = int(s // 60)
        return f"{m}:{(s % 60):04.1f}".replace(".", ",")

    f.write("Período\tConteúdo\tSpeaker\n")
    for seg in segments:
        texto = seg["text"].strip().replace("-->", "")
        if not texto:
            continue
        f.write(
            f"{_fmt(seg['start'])} - {_fmt(seg['end'])}\t{texto}\t{seg.get('speaker', '')}\n"
        )


# ---------------------------------------------------------------------------
# Card de conclusão do Passo 1.1
# ---------------------------------------------------------------------------

clear_output(wait=True)
card_ok(
    "Passo 1 concluído!",
    "Funções internas carregadas. Continue para o <strong>Passo 2</strong>.",
)

---

# PASSO 2 - Enviar arquivo

In [ ]:
# @markdown Clique em ▶ **play** para escolher como deseja enviar seus arquivos (até 3 de uma vez).

verificar_etapas("2a")  # Mantivemos a dependência do passo 1 que antes se chamava 2a/2b

if "batch_jobs" not in globals():
    globals()["batch_jobs"] = []

# Previne travamento inicial montando o output frame
# Loop de coleta
while True:
    output.clear(wait=True)
    display(HTML('<div id="attrindade-ui-container"></div>'))
    
    qtd_atual = len(globals()["batch_jobs"])
    
    # Gera lista de arquivos na fila
    lista_nomes = "".join([f"<li>{j['filename']}</li>" for j in globals()["batch_jobs"]])
    lista_html = f'<ul style="margin:5px 0 0 20px;padding:0;font-size:13px;color:#475569">{lista_nomes}</ul>' if qtd_atual > 0 else ""

    if qtd_atual >= 3:
        logger.info("[Passo 2] Limite de 3 arquivos atingido.")
        card_info(
            "Limite de Arquivos Atingido", 
            f"Você já selecionou 3 arquivos para processamento em lote:<br>{lista_html}<br>Prossiga para o Passo 3 para iniciar."
        )
        break

    js_escolha = f"""
    (async () => {{
      let container = document.getElementById('attrindade-ui-container');
      if(!container) {{
        container = document.createElement('div');
        container.id = 'attrindade-ui-container';
        document.body.appendChild(container);
      }}
      container.innerHTML = '';
      
      const qtdAtual = {qtd_atual};
      let avisoFila = "";
      if(qtdAtual > 0) {{
          avisoFila = `<div style="margin-bottom:16px;padding:12px;background:#e2e8f0;border-radius:8px;font-size:13px;color:#334155">
            <strong>Arquivos na fila (${{qtdAtual}}/3):</strong>
            {lista_html}
          </div>`;
      }}
      
      let btnProsseguir = qtdAtual > 0 
          ? `<button id="btn-finish" style="background:#059669;color:#fff;border:none;border-radius:8px;padding:12px 20px;font-size:15px;font-weight:600;cursor:pointer;display:flex;align-items:center;transition:background 0.2s;margin-top:10px">✅&nbsp; Finalizar e Ir para o Passo 3</button>`
          : '';

      const wrapper = document.createElement('div');
      wrapper.style.cssText = (
        'font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;' +
        'margin:10px 0;background:#f8fafc;border-left:4px solid #3b82f6;' +
        'border-radius:10px;padding:18px 22px;max-width:600px'
      );
     
      wrapper.innerHTML = (
        '<p style="margin:0 0 4px;font-size:17px;font-weight:700;color:#1e293b">📂 Adicionar Arquivo (' + qtdAtual + '/3)</p>'
        + '<p style="margin:0 0 16px;font-size:14px;color:#475569">Escolha uma opção para colocar na fila de processamento:</p>'
        + avisoFila
        + '<div style="display:flex;flex-direction:column;gap:10px">'
        + '<button id="btn-upload" style="background:#4f46e5;color:#fff;border:none;border-radius:8px;padding:12px 20px;font-size:15px;font-weight:600;cursor:pointer;display:flex;align-items:center;transition:background 0.2s">💻&nbsp; Fazer Upload do meu Computador</button>'
        + '<button id="btn-youtube" style="background:#fff;color:#1e293b;border:1px solid #cbd5e1;border-radius:8px;padding:12px 20px;font-size:15px;font-weight:600;cursor:pointer;display:flex;align-items:center;transition:background 0.2s">🔴&nbsp; Colar link do YouTube / Outros sites</button>'
        + '<button id="btn-gdrive" style="background:#fff;color:#1e293b;border:1px solid #cbd5e1;border-radius:8px;padding:12px 20px;font-size:15px;font-weight:600;cursor:pointer;display:flex;align-items:center;transition:background 0.2s">☁️&nbsp; Colar link do Google Drive</button>'
        + '</div>'
        + btnProsseguir
      );
      container.appendChild(wrapper);
     
      return new Promise(function(resolve){{
        wrapper.querySelector('#btn-upload').onclick = function(){{ container.innerHTML = ''; resolve('upload'); }};
        wrapper.querySelector('#btn-youtube').onclick = function(){{ container.innerHTML = ''; resolve('YouTube / Link'); }};
        wrapper.querySelector('#btn-gdrive').onclick = function(){{ container.innerHTML = ''; resolve('Google Drive'); }};
        if(qtdAtual > 0) {{
           wrapper.querySelector('#btn-finish').onclick = function(){{ container.innerHTML = ''; resolve('finish'); }};
        }}
      }});
    }})()
    """
    
    opcao = output.eval_js(js_escolha)
    if opcao == "finish":
        break

    target_file = None
    target_filename = None

    if opcao == "upload":
        card_info(
            "Botão Gerado: Fazer Upload Local",
            'Clique em <strong>"Escolher arquivos"</strong> abaixo (ou selecione o botão de upload que apareceu), '
            "selecione o arquivo no seu computador e aguarde a barra de progresso terminar.",
        )
        logger.info("[Passo 2] Iniciado: upload local.")

        uploaded = files.upload()
        if not uploaded:
            logger.warning("Nenhum arquivo enviado localmente.")
            continue

        target_filename = list(uploaded.keys())[0]
        size_mb = len(uploaded[target_filename]) / (1024 * 1024)
        target_file = move_para_pasta_input(target_filename)
        logger.info(
            f"[Passo 2] Arquivo recebido via Upload: '{target_filename}' ({size_mb:.1f} MB)"
        )

    else:
        logger.info(f"[Passo 2] Iniciado: download via {opcao}.")
        target_file, target_filename = insira_link(opcao)

        if target_file == "__VOLTAR__":
            continue

        size_mb = os.path.getsize(target_file) / (1024 * 1024)
        logger.info(
            f"[Passo 2] Arquivo recebido via {opcao}: '{target_filename}' ({size_mb:.1f} MB)"
        )
        
    if target_file and target_filename:
        globals()["batch_jobs"].append({
            "target": target_file, 
            "filename": target_filename,
            "transcription_result": None,
            "diarization_result": None
        })
        # Limpa os cards de "Aguarde" antes de mostrar o sucesso
        output.clear(wait=True)
        card_ok(
            f"Arquivo adicionado! ({len(globals()['batch_jobs'])}/3)",
            f"<strong>{target_filename}</strong> &nbsp;·&nbsp; (~ {size_mb:.1f} MB)",
            meta="Adicione mais ou clique em Finalizar para ir para o Passo 3."
        )

if len(globals()["batch_jobs"]) > 0:
    globals()["etapas_concluidas"].add("2a")
    globals()["etapas_concluidas"].add("2b")
    
    # Opcional: mostrar resumo dos arquivos no card final
    lista_final = "<br>".join([f"• {j['filename']}" for j in globals()["batch_jobs"]])
    card_ok(
        "Passo 2 Concluído!",
        f"Você selecionou {len(globals()['batch_jobs'])} arquivo(s):<br><strong>{lista_final}</strong>",
        meta=f"Por favor, desça e execute o Passo 3 para iniciar o processamento. · {obter_timestamp_brasil()}"
    )

---

# PASSO 3 - Transcrição

In [ ]:
# @markdown Clique em ▶ **play**

idioma_padrao = "Português 🇧🇷"

_IDIOMA_MAP = {
    "Português 🇧🇷": "pt",
    "Inglês 🇺🇸": "en",
    "Espanhol 🇪🇸": "es",
    "Francês 🇫🇷": "fr",
    "Alemão 🇩🇪": "de",
    "Italiano 🇮🇹": "it",
    "Japonês 🇯🇵": "ja",
    "Chinês 🇨🇳": "zh",
    "Árabe 🇸🇦": "ar",
    "Russo 🇷🇺": "ru",
    "Detectar automaticamente 🔍": None,
}
language = _IDIOMA_MAP[idioma_padrao]

import sys
import types
import torchaudio

if not hasattr(torchaudio, "AudioMetaData"):
    torchaudio.AudioMetaData = type("AudioMetaData", (), {})
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = lambda backend: None
if not hasattr(torchaudio, "get_audio_backend"):
    torchaudio.get_audio_backend = lambda: "soundfile"

if "torchaudio.backend" not in sys.modules:
    _backend_mod = types.ModuleType("torchaudio.backend")
    _common_mod = types.ModuleType("torchaudio.backend.common")
    _common_mod.AudioMetaData = torchaudio.AudioMetaData
    _backend_mod.common = _common_mod
    sys.modules["torchaudio.backend"] = _backend_mod
    sys.modules["torchaudio.backend.common"] = _common_mod
    torchaudio.backend = _backend_mod

verificar_etapas("3.1")
batch_jobs = globals().get("batch_jobs", [])
if not batch_jobs:
    raise RuntimeError(
        "Nenhum arquivo na fila! Execute o Passo 2 para selecionar arquivos."
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
batch_size = 16 if device == "cuda" else 4

logger.info(f"[Passo 3] Iniciado processamento em lote. ({len(batch_jobs)} arquivo(s))")
logger.info(f"[Passo 3] Idioma Padrão: {idioma_padrao}")

# ---------------------------------------------------------------------------
# INTERFACE DE SELEÇÃO POR ARQUIVO: IDIOMA E DIARIZAÇÃO
# ---------------------------------------------------------------------------
options_html = ""
for name, code in _IDIOMA_MAP.items():
    val = "None" if code is None else code
    sel = "selected" if code == language else ""
    options_html += f'<option value="{val}" {sel}>{name}</option>'

lista_selecao = ""
for i, job in enumerate(batch_jobs):
    fname = job["filename"]
    lista_selecao += f"""
    <div style="margin-bottom:12px; background:#fff; padding:14px; border-radius:8px; border:1px solid #cbd5e1; box-shadow:0 1px 2px rgba(0,0,0,0.05)">
        <label style="display:block; font-size:14px; font-weight:700; color:#334155; margin-bottom:8px;">📄 {fname}</label>

        <div style="display:flex; flex-direction:column; gap:10px;">
            <div>
                <label style="font-size:13px; color:#475569; margin-bottom:4px; display:block">Idioma do áudio:</label>
                <select id="lang_{i}" style="width:100%; padding:8px; border-radius:6px; border:1px solid #cbd5e1; font-size:14px; cursor:pointer; color:#1e293b; background:#f8fafc">
                    {options_html}
                </select>
            </div>

            <div style="display:flex; align-items:center; margin-top:4px;">
                <input type="checkbox" id="chk_diar_{i}" style="cursor:pointer;scale:1.2;margin-right:8px;" onchange="document.getElementById('spk_box_{i}').style.display = this.checked ? 'block' : 'none'">
                <label for="chk_diar_{i}" style="cursor:pointer;font-size:13px;font-weight:600;color:#1e293b">Separar locutores (Identificar quem fala)</label>
            </div>

            <div id="spk_box_{i}" style="display:none; padding-left:24px;">
                <label style="font-size:13px; color:#475569; margin-bottom:4px; display:block">Quantas pessoas falam?</label>
                <select id="spk_{i}" style="width:100%; padding:6px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fcfcfc">
                    <option value="None">Não sei</option>
                    <option value="1">1 pessoa</option>
                    <option value="2">2 pessoas</option>
                    <option value="3">3 pessoas</option>
                    <option value="4">4 pessoas</option>
                    <option value="5">5 pessoas</option>
                    <option value="6">6 pessoas</option>
                    <option value="7">7 pessoas</option>
                    <option value="8">8 pessoas</option>
                    <option value="9">9 pessoas</option>
                    <option value="10">10 pessoas</option>
                </select>
            </div>
        </div>
    </div>
    """

js_escolha_lang = f"""
(async () => {{
    let container = document.getElementById('attrindade-ui-container');
    if(!container) {{
        container = document.createElement('div');
        container.id = 'attrindade-ui-container';
        document.body.appendChild(container);
    }}
    container.innerHTML = '';

    const wrapper = document.createElement('div');
    wrapper.style.cssText = (
    'font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;' +
    'margin:10px 0;background:#f8fafc;border-left:4px solid #4f46e5;' +
    'border-radius:10px;padding:20px 24px;max-width:600px;box-shadow:0 4px 6px -1px rgb(0 0 0 / 0.1)'
    );

    wrapper.innerHTML = (
    '<p style="margin:0 0 4px;font-size:18px;font-weight:800;color:#1e293b">⚙️ Configurações do Lote</p>'
    + '<p style="margin:0 0 16px;font-size:14px;color:#475569">Defina o idioma e as opções de separação de locutores para cada arquivo.</p>'
    + `<div style="margin-bottom:20px;">{lista_selecao}</div>`
    + '<button id="btn-start-trans" style="background:#4f46e5;color:#fff;border:none;border-radius:8px;padding:12px 24px;font-size:15px;font-weight:700;cursor:pointer;transition:all 0.2s;width:100%;box-shadow:0 1px 3px rgba(0,0,0,0.1)">🚀&nbsp; Iniciar Processamento</button>'
    );
    container.appendChild(wrapper);

    return new Promise(function(resolve){{
    wrapper.querySelector('#btn-start-trans').onclick = function(){{
        let configs = [];
        for (let i = 0; i < {len(batch_jobs)}; i++) {{
            let lang = wrapper.querySelector('#lang_' + i).value;
            let diar = wrapper.querySelector('#chk_diar_' + i).checked;
            let spks = wrapper.querySelector('#spk_' + i).value;
            configs.push(JSON.stringify({{lang: lang, diar: diar, spks: spks}}));
        }}
        container.innerHTML = '';
        resolve(configs.join('|||'));
    }};
    }});
}})()
"""

display(HTML('<div id="attrindade-ui-container"></div>'))
res_configs_str = output.eval_js(js_escolha_lang)

needs_diarization = False

if res_configs_str:
    import json

    configs_list = res_configs_str.split("|||")
    for i, c in enumerate(configs_list):
        conf = json.loads(c)
        batch_jobs[i]["selected_language"] = (
            None if conf["lang"] == "None" else conf["lang"]
        )
        batch_jobs[i]["run_diarization"] = conf["diar"]
        batch_jobs[i]["min_speakers"] = (
            None if conf["spks"] == "None" else int(conf["spks"])
        )
        if conf["diar"]:
            needs_diarization = True
else:
    for job in batch_jobs:
        job["selected_language"] = language
        job["run_diarization"] = False
        job["min_speakers"] = None

output.clear()
logger.info(f"[Passo 3] Dispositivo: {device} | compute_type: {compute_type}")

# ---------------------------------------------------------------------------
# 1. Carregar Modelo Principal WhisperX
# ---------------------------------------------------------------------------
display(
    HTML(
        '<div id="load-msg-model" style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',Roboto,sans-serif;'
        'margin:10px 0;background:#fffbeb;border-left:4px solid #f59e0b;border-radius:10px;padding:14px 18px">'
        '<p style="margin:0 0 4px;font-size:16px;font-weight:700;color:#1e293b">⏳&nbsp; Carregando IA (Whisper)…</p>'
        '<p style="margin:0;font-size:13px;color:#d97706;font-style:italic">Esse passo pode demorar em torno de um minuto na primeira vez. Por favor, aguarde.</p>'
        "</div>"
    )
)

logger.info("(1/4) Carregando modelo base...")

# Se todos os arquivos utilizam o mesmo idioma, injetamos diretamente no load_model
_idiomas_lote = {
    j["selected_language"] for j in batch_jobs if j.get("selected_language")
}
_idioma_global = _idiomas_lote.pop() if len(_idiomas_lote) == 1 else None

with Silenciador():
    model = whisperx.load_model(
        "large-v3-turbo",
        device,
        compute_type=compute_type,
        vad_method="silero",
        language=_idioma_global,
    )

output.eval_js('document.getElementById("load-msg-model")?.remove()')

# ---------------------------------------------------------------------------
# 2. Processar Transcrição de todos os arquivos
# ---------------------------------------------------------------------------
for idx, job in enumerate(batch_jobs):
    target = job["target"]
    target_filename = job["filename"]

    # O logger de início foi movido mais para baixo para incluir a duração

    with Silenciador():
        audio = whisperx.load_audio(target)
    _dur_s = int(len(audio) / 16000)

    def _fmt_tempo(s):
        if s < 60:
            return f"{s}s"
        h, rem = divmod(s, 3600)
        m, s = divmod(rem, 60)
        if h:
            return f"{h}h {m}min {s}s"
        return f"{m}min {s}s"

    _dur_fmt = _fmt_tempo(_dur_s)
    _dur_min = _dur_s / 60.0
    # Estimativas calibradas (GPU): 28.6s fixo (1a execução) + ~1.99s/min para transcrição
    # CPU: ~2.5x mais lento
    _CUSTO_TRANS_S_MIN = 2.1 if device == "cuda" else 5.0
    _CUSTO_ALIGN_S_MIN = 2.28 if device == "cuda" else 5.7
    _est_trans_s = max(5, int(_dur_min * _CUSTO_TRANS_S_MIN))
    _est_align_s = max(5, int(_dur_min * _CUSTO_ALIGN_S_MIN)) + 30

    logger.info(
        f"Processando Transcrição {idx+1}/{len(batch_jobs)}: {target_filename} (Duração do áudio: {_dur_fmt})"
    )

    import time

    _trans_bar_id = f"wb-prog-trans-{idx}"
    _align_bar_id = f"wb-prog-align-{idx}"

    _js_timer = """
    (function(){
      var start  = Date.now();
      function fmt(s) {
        if (s < 60) return s + 's';
        var m = Math.floor(s / 60), r = s % 60;
        return r > 0 ? m + 'min ' + r + 's' : m + 'min';
      }
      var iv = setInterval(function(){
        var elEl = document.getElementById('REPLACE_ME_ID');
        if (!elEl) { clearInterval(iv); return; }
        var elapsed = Math.floor((Date.now() - start) / 1000);
        elEl.textContent = 'Decorrido: ' + fmt(elapsed);
      }, 1000);
    })();
    """.replace(
        "REPLACE_ME_ID", f"wb-trans-elapsed-{idx}"
    )

    display(
        HTML(
            '<div id="wb-trans-card" style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\','
            "Roboto,sans-serif;margin:10px 0;background:#f0f9ff;border-left:4px solid #0284c7;"
            'border-radius:10px;padding:16px 20px">'
            f'<p style="margin:0 0 12px;font-size:17px;font-weight:700;color:#1e293b">⏳&nbsp; Transcrevendo arquivo {idx+1} de {len(batch_jobs)}…</p>'
            '<table style="border-collapse:collapse;font-size:13px;width:100%;margin-bottom:12px">'
            f'<tr><td style="padding:3px 14px 3px 0;color:#94a3b8;font-weight:600;white-space:nowrap">Arquivo</td>'
            f'<td style="padding:3px 0;color:#1e293b;font-weight:600">{target_filename}</td></tr>'
            f'<tr><td style="padding:3px 14px 3px 0;color:#94a3b8;font-weight:600">Duração</td>'
            f'<td style="padding:3px 0;color:#1e293b">{_dur_fmt}</td></tr>'
            '<tr><td style="padding:3px 14px 3px 0;color:#94a3b8;font-weight:600">Estimativa</td>'
            f'<td style="padding:3px 0;color:#1e293b">~{_fmt_tempo(_est_trans_s)}</td></tr>'
            "</table>"
            '<div style="display:flex;justify-content:space-between;font-size:12px;color:#64748b;margin-top:2px">'
            f'<span id="wb-trans-elapsed-{idx}">Decorrido: 0s</span>'
            "<span>Processando...</span>"
            "</div>"
            f"<script>{_js_timer}</script>"
            "</div>"
        )
    )
    display(HTML("<div></div>"), display_id=_trans_bar_id)

    logger.info("Transcrevendo...")
    _t0_trans = time.time()
    with Silenciador(display_id=_trans_bar_id):
        result = model.transcribe(
            audio,
            batch_size=batch_size,
            print_progress=True,
            language=job["selected_language"],
        )
    _t1_trans = time.time()
    logger.info(
        f"Tempo de transcrição ({target_filename}): {_t1_trans - _t0_trans:.2f}s"
    )

    output.clear()

    # Salva o idioma usado (importante para o alinhamento e para o estado do job)
    idioma_final = result.get("language", job["selected_language"])

    _js_timer_align = """
    (function(){
      var start  = Date.now();
      function fmt(s) {
        if (s < 60) return s + 's';
        var m = Math.floor(s / 60), r = s % 60;
        return r > 0 ? m + 'min ' + r + 's' : m + 'min';
      }
      var iv = setInterval(function(){
        var elEl = document.getElementById('REPLACE_ME_ID');
        if (!elEl) { clearInterval(iv); return; }
        var elapsed = Math.floor((Date.now() - start) / 1000);
        elEl.textContent = 'Decorrido: ' + fmt(elapsed);
      }, 1000);
    })();
    """.replace(
        "REPLACE_ME_ID", f"wb-align-elapsed-{idx}"
    )

    # Alinhamento
    display(
        HTML(
            "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
            'margin:10px 0;background:#f0f9ff;border-left:4px solid #0284c7;border-radius:10px;padding:14px 18px">'
            f'<p style="margin:0 0 8px;font-size:16px;font-weight:700;color:#1e293b">⏳&nbsp; Alinhando timestamps ({idx+1}/{len(batch_jobs)})…</p>'
            '<table style="border-collapse:collapse;font-size:13px;width:100%;margin-bottom:10px">'
            f'<tr><td style="padding:3px 14px 3px 0;color:#94a3b8;font-weight:600;white-space:nowrap">Arquivo</td><td style="color:#1e293b;font-weight:600">{target_filename}</td></tr>'
            f'<tr><td style="padding:3px 14px 3px 0;color:#94a3b8;font-weight:600">Estimativa</td><td style="color:#1e293b">~{_fmt_tempo(_est_align_s)}</td></tr>'
            "</table>"
            '<div style="display:flex;justify-content:space-between;font-size:12px;color:#64748b;margin-top:2px">'
            f'<span id="wb-align-elapsed-{idx}">Decorrido: 0s</span>'
            "<span>Processando...</span>"
            "</div>"
            f"<script>{_js_timer_align}</script>"
            "</div>"
        )
    )
    display(HTML("<div></div>"), display_id=_align_bar_id)

    logger.info("Alinhando timestamps...")
    _t0_align = time.time()
    with Silenciador(display_id=_align_bar_id):
        model_a, metadata = whisperx.load_align_model(
            language_code=idioma_final, device=device
        )
        result = whisperx.align(
            result["segments"],
            model_a,
            metadata,
            audio,
            device,
            return_char_alignments=False,
            print_progress=True,
        )
    _t1_align = time.time()
    logger.info(
        f"Tempo de alinhamento ({target_filename}): {_t1_align - _t0_align:.2f}s"
    )

    del model_a
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    job["transcription_result"] = result["segments"]
    job["language"] = idioma_final
    output.clear()

del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# ---------------------------------------------------------------------------
# 3. Processar Diarização (Apenas os marcados)
# ---------------------------------------------------------------------------
if needs_diarization:
    logger.info("(2/4) Carregando modelo Pyannote para diarização...")
    display(
        HTML(
            '<div id="load-msg-pyannote" style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',Roboto,sans-serif;'
            'margin:10px 0;background:#fffbeb;border-left:4px solid #f59e0b;border-radius:10px;padding:14px 18px">'
            '<p style="margin:0 0 4px;font-size:16px;font-weight:700;color:#1e293b">⏳&nbsp; Carregando IA de Separação de Vozes…</p>'
            "</div>"
        )
    )

    with Silenciador():
        import base64
        import torch
        from huggingface_hub import login
        import pyannote.audio.core.task
        import pyannote.core

        torch.serialization.add_safe_globals(
            [
                torch.torch_version.TorchVersion,
                pyannote.audio.core.task.Specifications,
                pyannote.audio.core.task.Problem,
                pyannote.audio.core.task.Resolution,
            ]
        )
        pyannote.core.Annotation.speaker_diarization = property(lambda self: self)

        _chave = base64.b64decode(
            b"aGZfVWdWZElEZEpLYWJDUkFyaXlIRkhRTmRKQWR1SmZ0UHFKQw=="
        ).decode("utf-8")
        login(token=_chave, add_to_git_credential=False)

        diarize_model = whisperx.diarize.DiarizationPipeline(
            model_name="pyannote/speaker-diarization-3.1", token=_chave, device=device
        )

    output.eval_js('document.getElementById("load-msg-pyannote")?.remove()')

    import re
    import time

    # Dicionário para armazenar trechos agrupados para dar os nomes DEPOIS de todos concluídos
    batch_diarization_results = {}

    for idx, job in enumerate(batch_jobs):
        if not job["run_diarization"]:
            continue

        target = job["target"]
        target_filename = job["filename"]
        min_spks = job["min_speakers"]
        _job_dur_s = int(len(whisperx.load_audio(target)) / 16000)
        _job_dur_min = _job_dur_s / 60.0
        _CUSTO_DIAR_S_MIN = 2.78 if device == "cuda" else 7.0
        _est_diar_s = max(5, int(_job_dur_min * _CUSTO_DIAR_S_MIN))
        _job_dur_fmt = _fmt_tempo(_job_dur_s)
        _diar_bar_id = f"wb-prog-diar-{idx}"

        _js_timer_diar = """
        (function(){
          var start  = Date.now();
          function fmt(s) {
            if (s < 60) return s + 's';
            var m = Math.floor(s / 60), r = s % 60;
            return r > 0 ? m + 'min ' + r + 's' : m + 'min';
          }
          var iv = setInterval(function(){
            var elEl = document.getElementById('REPLACE_ME_ID');
            if (!elEl) { clearInterval(iv); return; }
            var elapsed = Math.floor((Date.now() - start) / 1000);
            elEl.textContent = 'Decorrido: ' + fmt(elapsed);
          }, 1000);
        })();
        """.replace(
            "REPLACE_ME_ID", f"wb-diar-elapsed-{idx}"
        )

        display(
            HTML(
                '<div style=\'font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;'
                "margin:10px 0;background:#fdf4ff;border-left:4px solid #a855f7;border-radius:10px;padding:14px 18px'>"
                f"<p style='margin:0 0 8px;font-size:16px;font-weight:700;color:#1e293b'>🗣️&nbsp; Separando vozes ({idx+1}/{len(batch_jobs)})…</p>"
                "<table style='border-collapse:collapse;font-size:13px;width:100%;margin-bottom:10px'>"
                f"<tr><td style='padding:3px 14px 3px 0;color:#94a3b8;font-weight:600;white-space:nowrap'>Arquivo</td><td style='color:#1e293b;font-weight:600'>{target_filename}</td></tr>"
                f"<tr><td style='padding:3px 14px 3px 0;color:#94a3b8;font-weight:600'>Duração</td><td style='color:#1e293b'>{_job_dur_fmt}</td></tr>"
                f"<tr><td style='padding:3px 14px 3px 0;color:#94a3b8;font-weight:600'>Estimativa</td><td style='color:#1e293b'>~{_fmt_tempo(_est_diar_s)}</td></tr>"
                "</table>"
                "<p style='margin:5px 0 0;font-size:12px;color:#64748b'>Por favor, não feche esta aba enquanto processa.</p>"
                '<div style="display:flex;justify-content:space-between;font-size:12px;color:#64748b;margin-top:8px">'
                f'<span id="wb-diar-elapsed-{idx}">Decorrido: 0s</span>'
                "<span>Processando...</span>"
                "</div>"
                f"<script>{_js_timer_diar}</script>"
                "</div>"
            )
        )
        display(HTML("<div></div>"), display_id=_diar_bar_id)

        logger.info(f"Diarizando {target_filename}...")
        _t0_diar = time.time()
        with Silenciador(display_id=_diar_bar_id):
            audio = whisperx.load_audio(target)
            diarize_segments = diarize_model(audio, min_speakers=min_spks)

            res_dict = {
                "segments": job["transcription_result"],
                "language": job["language"],
            }
            res_dict = whisperx.assign_word_speakers(diarize_segments, res_dict)
        _t1_diar = time.time()
        logger.info(
            f"Tempo de separação de vozes ({target_filename}): {_t1_diar - _t0_diar:.2f}s"
        )

        output.clear()
        time.sleep(0.5)

        ssm = res_dict["segments"]

        ssm_limpo = []
        for s in ssm:
            texto_limpo = re.sub(r'["\'\`\\\n\r]', " ", s.get("text", ""))
            ssm_limpo.append(
                {
                    "start": s.get("start", 0),
                    "end": s.get("end", 0),
                    "speaker": s.get("speaker", "SPEAKER_00"),
                    "text": texto_limpo,
                }
            )

        # Salva para uso posterior (nomeação e unificador final)
        job["diarization_temp"] = ssm
        batch_diarization_results[target_filename] = ssm_limpo

    del diarize_model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    # ---------------------------------------------------------------------------
    # 4. Nomear locutores em batch (Assíncrono)
    # ---------------------------------------------------------------------------
    if batch_diarization_results:
        name_speakers_async(batch_diarization_results, batch_jobs)

logger.info(f"[Passo 3] Concluído para todos os {len(batch_jobs)} arquivos.")
globals()["etapas_concluidas"].add("3.1")
globals()["etapas_concluidas"].add(
    "3.2"
)  # Marcamos como 3.2 tbm pro passo 4 saber que está ok

if not needs_diarization:
    display(
        HTML(
            "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
            'margin:10px 0;background:#d1fae5;border-left:4px solid #059669;border-radius:10px;padding:16px 20px">'
            f'<p style="margin:0 0 6px;font-size:17px;font-weight:700;color:#1e293b">✅&nbsp; Processamento concluído ({len(batch_jobs)} arquivo(s))!</p>'
            f'<p style="margin:0 0 14px;font-size:14px;color:#475569">{obter_timestamp_brasil()}</p>'
            '<div style="background:#fff;border:1px solid #a7f3d0;border-radius:8px;padding:12px 16px">'
            '<p style="margin:0 0 4px;font-size:14px;font-weight:700;color:#1e293b">Vá para o <strong>Passo 4</strong> para baixar os textos gerados.</p>'
            "</div>"
            "</div>"
        )
    )

---

# PASSO 4 - Download

In [ ]:
# @markdown Clique em ▶ play para habilitar as opções de download.

import base64
import os

verificar_etapas(4)
batch_jobs = globals().get("batch_jobs", [])
com_diar = "3.2" in globals().get("etapas_concluidas", set())


def get_b64_uri(filepath):
    with open(filepath, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode("utf-8")
    return f"data:application/octet-stream;charset=utf-8;base64,{b64}"


def btn(title, filepath, icone="📄", cor="#4f46e5"):
    filename = os.path.basename(filepath)
    uri = get_b64_uri(filepath)
    return (
        f'<a href="{uri}" download="{filename}" target="_blank" rel="noopener noreferrer" '
        f'style="display:inline-flex;align-items:center;'
        f"padding:8px 14px;background:{cor};color:#fff;text-decoration:none;border-radius:8px;"
        f"font-size:13px;font-weight:600;margin:0 8px 8px 0;box-shadow:0 1px 2px rgba(0,0,0,0.1);"
        f'transition:background 0.2s">{icone}&nbsp; {title}</a>'
    )


todos_botoes = ""

for job in batch_jobs:
    target_filename = job["filename"]
    basename, _ = os.path.splitext(target_filename)
    base_path = os.path.join("resultados", basename)
    os.makedirs("resultados", exist_ok=True)

    job_has_diar = bool(job.get("diarization_result"))

    if job_has_diar:
        ssm_mod = job["diarization_result"]
    else:
        ssm_mod = job.get("transcription_result", [])

    if not ssm_mod:
        continue  # Should not happen if step 3 was completed

    # --- Gerar TXT ---
    with open(f"{base_path}.txt", "w", encoding="utf-8-sig") as f:
        write_txt(ssm_mod, f)
    with open(f"{base_path}_tempolinha.txt", "w", encoding="utf-8-sig") as f:
        write_txt(ssm_mod, f, timestamp_linha=True)

    if job_has_diar:
        with open(f"{base_path}_loc.txt", "w", encoding="utf-8-sig") as f:
            write_txt(ssm_mod, f, locutores=True)
        with open(f"{base_path}_loc_tempolinha.txt", "w", encoding="utf-8-sig") as f:
            write_txt(ssm_mod, f, locutores=True, timestamp_linha=True)

    # --- Gerar SRT (Legenda) ---
    with open(f"{base_path}.srt", "w", encoding="utf-8-sig") as f:
        write_srt(ssm_mod, f)

    if job_has_diar:
        with open(f"{base_path}_loc.srt", "w", encoding="utf-8-sig") as f:
            write_srt(ssm_mod, f, locutores=True)

    # --- Gerar NVivo ---
    with open(f"{base_path}_nvivo.txt", "w", encoding="utf-8-sig") as f:
        write_nvivo_tabbed(ssm_mod, f)

    botoes_html = ""
    botoes_html += btn(
        "Texto Simples (.txt)", f"{base_path}.txt", icone="📝", cor="#3b82f6"
    )
    botoes_html += btn(
        "Texto c/ Tempo (.txt)", f"{base_path}_tempolinha.txt", icone="⏱️", cor="#3b82f6"
    )
    botoes_html += btn("Legenda (.srt)", f"{base_path}.srt", icone="🎬", cor="#8b5cf6")
    botoes_html += btn(
        "NVivo (.txt)", f"{base_path}_nvivo.txt", icone="📊", cor="#10b981"
    )

    botoes_diar_html = ""
    if job_has_diar:
        botoes_diar_html += "<div style='margin-bottom:8px;'>"
        botoes_diar_html += btn(
            "Locutores (.txt)", f"{base_path}_loc.txt", icone="👥", cor="#3b82f6"
        )
        botoes_diar_html += btn(
            "Locutores c/ Tempo (.txt)",
            f"{base_path}_loc_tempolinha.txt",
            icone="⏱️",
            cor="#3b82f6",
        )
        botoes_diar_html += btn(
            "Legenda Locutores (.srt)",
            f"{base_path}_loc.srt",
            icone="🎬",
            cor="#8b5cf6",
        )
        botoes_diar_html += "</div>"

    todos_botoes += f"<div style='margin-bottom:15px;padding:10px;background:#fff;border-radius:8px;border:1px solid #e2e8f0;'>"
    todos_botoes += f"<p style='margin:0 0 10px;font-size:14px;font-weight:700;color:#334155'>📄 {target_filename}</p>"
    todos_botoes += f"<div style='margin-bottom:8px'>{botoes_html}</div>"
    todos_botoes += botoes_diar_html
    todos_botoes += "</div>"

logger.info(f"[Passo 4] Exportação concluída.")
globals()["etapas_concluidas"].add(4)

display(
    HTML(
        "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
        'margin:10px 0;background:#f0fdfa;border-left:4px solid #14b8a6;border-radius:10px;padding:16px 20px">'
        '<p style="margin:0 0 16px;font-size:17px;font-weight:700;color:#1e293b">✅&nbsp; Arquivos Prontos para Download!</p>'
        f"{todos_botoes}"
        '<div style="margin-top:18px;padding:14px 16px;background:#fdfce8;border:1px solid #fde047;border-radius:8px;display:flex;align-items:flex-start;gap:12px">'
        '<span style="font-size:22px;line-height:1">☕</span>'
        "<div>"
        '<p style="margin:0 0 3px;font-size:13px;font-weight:700;color:#713f12">Gostou da ferramenta? Considere apoiar o projeto!</p>'
        '<p style="margin:0;font-size:12px;color:#92400e;line-height:1.5">Uma contribuição via Pix ajuda a manter essa iniciativa gratuita e em constante melhoria.<br>'
        '<strong>Chave Pix:</strong> <code style="background:#fef9c3;padding:1px 5px;border-radius:4px;font-size:12px">attrindade.dados@gmail.com</code></p>'
        "</div>"
        "</div>"
        "</div>"
    )
)

---

# Solução de Problemas

In [ ]:
# @markdown Se algo não funcionou, clique em ▶ **play** para baixar o arquivo de diagnóstico
# @markdown e envie-o para [**attrindade.dados@gmail.com**](mailto:attrindade.dados@gmail.com) descrevendo o que aconteceu.


logger.info("FIM DE SESSÃO")
logger.info(
    f"Etapas concluídas: {sorted(str(e) for e in globals()['etapas_concluidas'])}"
)
logger.complete()

files.download(f"/content/{LOG_FILE}")

display(
    HTML(
        "<div style=\"font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;"
        'margin:10px 0;background:#fee2e2;border-left:4px solid #dc2626;border-radius:10px;padding:16px 20px">'
        '<p style="margin:0 0 4px;font-size:17px;font-weight:700;color:#1e293b">📋&nbsp; Arquivo de diagnóstico baixado</p>'
        f'<p style="margin:0 0 12px;font-size:14px;color:#475569">Arquivo: <code style="background:#fecaca;'
        f'padding:2px 6px;border-radius:4px;font-size:13px">{LOG_FILE}</code></p>'
        '<div style="background:#fff;border:1px solid #fca5a5;border-radius:8px;padding:12px 16px">'
        '<p style="margin:0 0 6px;font-size:13px;font-weight:700;color:#1e293b">Envie para:</p>'
        '<p style="margin:0 0 8px;font-size:14px;font-weight:600"><a href="mailto:attrindade.dados@gmail.com" style="color:#dc2626;text-decoration:none">📧 attrindade.dados@gmail.com</a></p>'
        '<p style="margin:0;font-size:13px;color:#475569">'
        "Descreva no e-mail em qual passo parou e qual mensagem apareceu. "
        "Quanto mais detalhes, mais rápido conseguimos resolver!</p>"
        "</div>"
        "</div>"
    )
)